# 03. 최고 모델을 레지스트리에 올리고 리니지와 재현성 확인하기

책 4.7절. 점수가 가장 높은 run의 모델을 `income-classifier`로 등록하고 alias `champion`을 붙입니다.
5장 파이프라인과 6장 서빙은 run ID가 아니라 `models:/income-classifier@champion`으로 모델을 꺼냅니다. alias를 옮기면 소비하는 쪽 코드를 바꾸지 않고 모델을 바꿀 수 있습니다.
마지막으로 등록된 모델에서 출발해 run, 파라미터, 데이터셋을 거꾸로 따라가고, 같은 데이터로 다시 학습해 점수가 같은지 확인합니다.

In [ ]:
import mlflow
from mlflow import MlflowClient
from sklearn.metrics import roc_auc_score
from income import download_df

client = MlflowClient()
MODEL_NAME = "income-classifier"
exp = client.get_experiment_by_name("income-classifier")

In [ ]:
best = client.search_runs([exp.experiment_id], filter_string="metrics.roc_auc_score_test > 0.8",
                          order_by=["metrics.roc_auc_score_test DESC"], max_results=1)[0]
print(best.info.run_name, best.info.run_id, round(best.data.metrics["roc_auc_score_test"], 4))

# MLflow 3에서는 runs:/<run_id>/<name> 대신 run에 남긴 LoggedModel URI(models:/m-...)로 등록한다.
mv = mlflow.register_model(best.data.tags["model_uri"], MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f"registered {MODEL_NAME} v{mv.version} @champion")

In [ ]:
# 리니지: alias 하나에서 출발해 run, 파라미터, 데이터셋, 아티팩트를 거꾸로 따라간다.
import json
champ = client.get_model_version_by_alias(MODEL_NAME, "champion")
run = client.get_run(champ.run_id)
print("model   :", f"{MODEL_NAME} v{champ.version}", "<- run", run.info.run_name, champ.run_id)
print("metrics :", {k: round(v, 4) for k, v in run.data.metrics.items() if k.startswith("roc_auc")})
print("params  :", {k: run.data.params[k] for k in sorted(run.data.params) if k in ("random_state", "seed", "n_estimators", "max_depth")})
for d in run.inputs.dataset_inputs:
    context = {t.key: t.value for t in d.tags}.get("mlflow.data.context", "")
    print("dataset :", context.ljust(9), d.dataset.name.ljust(8), json.loads(d.dataset.source).get("uri"))
print("artifacts:", [a.path for a in client.list_artifacts(champ.run_id)])

In [ ]:
# 재현성: 기록된 데이터셋 경로에서 데이터를 다시 받아, 같은 모델 정의로 다시 학습한다.
sources = {d.dataset.name: json.loads(d.dataset.source).get("uri") for d in run.inputs.dataset_inputs}
train, test = download_df(sources["train"]), download_df(sources["test"])
X_tr, y_tr = train.drop(columns="Target"), train.Target
X_te, y_te = test.drop(columns="Target"), test.Target

model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
flavor = model.metadata.flavors
if "sklearn" in flavor:
    original = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
    retrained = type(original)(**original.get_params()).fit(X_tr, y_tr)
    score = roc_auc_score(y_te, retrained.predict_proba(X_te)[:, 1])
else:
    import xgboost as xgb
    params = {"objective": "binary:logistic", "max_depth": 10, "learning_rate": 0.3, "seed": 42}
    booster = xgb.train(params, xgb.DMatrix(X_tr, label=y_tr), num_boost_round=30)
    score = roc_auc_score(y_te, booster.predict(xgb.DMatrix(X_te)))

logged = run.data.metrics["roc_auc_score_test"]
print(f"logged={logged:.6f} retrained={score:.6f} same={abs(logged - score) < 1e-9}")